# load libraries

In [1]:
import numpy as np
import pandas as pd
from functools import lru_cache

# define game constraints
basic rules, numbers, and structures of the minigame, collected by manually recording data from each round

In [2]:
# possible user actions
actions = {
    "small": np.arange(1,21),
    "large": np.arange(20,46),
    "very_small": np.arange(2,10)
}

# round structures
round1 = [
    {"conv":None,"win":(70,100)},
    {"conv":(60,70),"win":(70,100)},
    {"conv":(60,70),"win":(70,100)},
    {"conv":(65,75),"win":(75,100)},
    {"conv":(65,75),"win":(75,100)},
    {"conv":(70,80),"win":(80,100)}
]

round2 = [
    {"conv":(60,70),"win":(70,100)},
    {"conv":(60,70),"win":(70,100)},
    {"conv":(65,75),"win":(75,100)},
    {"conv":(65,75),"win":(75,100)},
    {"conv":(70,80),"win":(80,100)},
    {"conv":(70,85),"win":(85,100)},
    {"conv":(75,85),"win":(85,100)},
    {"conv":(75,85),"win":(85,100)}
]

round3 = [
    {"conv":(65,75),"win":(75,100)},
    {"conv":(65,75),"win":(75,100)},
    {"conv":(70,80),"win":(80,100)},
    {"conv":(70,80),"win":(80,100)},
    {"conv":(75,85),"win":(85,100)},
    {"conv":(75,85),"win":(85,100)},
    {"conv":(80,90),"win":(90,100)},
    {"conv":(80,98),"win":(98,100)},
    {"conv":(90,95),"win":(95,100)},
    {"conv":(90,95),"win":(95,100)}
]



# define game assumptions
these could be empirically verified, but for simplicity, I make some initial assumptions

In [3]:
# assume values for each range of actions are uniformly distributed
action_probs = {
    a: np.ones(len(v))/len(v)
    for a,v in actions.items()
}

# assume probability of succesful convincing is 0.5
p_convince = 0.5

# functions for basic model implementation

In [10]:
def solve_round(round_rules):

    n_trials = len(round_rules)

    @lru_cache(None)

    def V(t, score, vs_left):

        if t == n_trials:
            return 1.0

        rule = round_rules[t]
        win_low, win_high = rule["win"]
        conv_range = rule["conv"]

        best = 0

        for action in ["small","large","very_small","convince"]:

            if action == "very_small" and vs_left == 0:
                continue

            if action == "convince":

                if conv_range and conv_range[0] <= score <= conv_range[1]:
                    best = max(best, p_convince)

                continue

            increments = actions[action]
            probs = action_probs[action]

            vs_next = vs_left - 1 if action == "very_small" else vs_left

            expected = 0

            for inc,p in zip(increments,probs):

                new_score = score + inc

                if new_score > 101:
                    val = 0

                elif win_low <= new_score <= win_high:
                    val = V(t+1, new_score, vs_next)

                else:
                    val = V(t, new_score, vs_next)

                expected += p * val

            best = max(best, expected)

        return min(1.0, max(0.0, best))

    return V

In [ ]:
def compute_policy(round_rules):

    V = solve_round(round_rules)
    n_trials = len(round_rules)

    policy = []

    for t in range(n_trials):
        for score in range(0,101):

            best_action = None
            best_val = -1

            for action in actions:

                val = V(t,score,3)

                if val > best_val:
                    best_val = val
                    best_action = action

            policy.append({
                "trial":t+1,
                "score":score,
                "best_action":best_action,
                "win_probability":best_val
            })

    return pd.DataFrame(policy)

# actual computation

In [9]:
policy_r1 = compute_policy(round1)
policy_r2 = compute_policy(round2)
policy_r3 = compute_policy(round3)

# TODO: finetune for the last three trials of 10... 